## 1 · Setup

Install the SDKs and authenticate. On Colab Enterprise / Vertex AI Workbench `google.auth.default()` picks up the attached service account automatically; locally you'll need `gcloud auth application-default login` first.

We use the **`google-genai`** SDK for generation and **`google-cloud-aiplatform[evaluation]`** for the Evaluation Service.

Required IAM role on the project: `roles/aiplatform.user`.

Required APIs (enable once per project, before running the notebook):

```
gcloud services enable aiplatform.googleapis.com
```

In [1]:
!pip install --upgrade --quiet \
    google-genai \
    "google-cloud-aiplatform[evaluation]" \
    pandas \
    pytest \
    ipytest

In [2]:
import google.auth

credentials, default_project = google.auth.default()

PROJECT_ID = default_project or "your-project-id"
LOCATION = "us-central1"

MODEL_ID = "gemini-2.5-flash"

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model:    {MODEL_ID}")

Project: qwiklabs-gcp-04-dc5ba3682c9e
Location: us-central1
Model:    gemini-2.5-flash


In [3]:
from google import genai
from google.genai import types

# Route the google-genai SDK at Vertex AI (rather than the public Gemini API).
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print("google-genai client ready (Vertex AI backend)")

google-genai client ready (Vertex AI backend)


## 2 · Function 1 — Question classifier (deterministic)

Classifies a citizen's question into exactly one of four categories. We constrain the output hard so the result is deterministic and can be checked with a plain `assertEqual`:

- The allowed labels are spelled out in the prompt.
- `temperature=0` removes sampling randomness.
- We post-process to map the raw text back onto one of the canonical labels, so stray punctuation or casing doesn't break the assertion.

In [4]:
CATEGORIES = ["Employment", "General Information", "Emergency Services", "Tax Related"]


def classify_question(question: str) -> str:
    """Classify a citizen question into one of CATEGORIES using Gemini.

    Deterministic task: temperature=0 and a constrained label set mean the
    same input always produces the same output, so it can be unit tested
    with assertEqual.
    """
    prompt = f"""You are a routing assistant for a city government help desk.
Classify the citizen's question into exactly ONE of these categories:
- Employment
- General Information
- Emergency Services
- Tax Related

Output ONLY the category name, exactly as written above. No other text.

Question: {question}
Category:"""

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=20,
            # gemini-2.5-flash is a thinking model: with a small output budget,
            # reasoning tokens can eat the whole allowance and leave .text=None.
            # Disable thinking for these short, deterministic calls.
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    raw = (response.text or "").strip()

    # Normalize the model output back onto a canonical label so minor
    # formatting differences don't break the deterministic assertion.
    for category in CATEGORIES:
        if category.lower() in raw.lower():
            return category
    return raw


# Quick smoke test
print(classify_question("How do I apply for a job with the city?"))
print(classify_question("There is a gas leak on Main Street, who do I call?"))
print(classify_question("When is my property tax payment due?"))
print(classify_question("What are the city hall hours?"))

Employment
Emergency Services
Tax Related
General Information


## 3 · Function 2 — Government social-media post generator (indeterminate)

Generates a short social-media post for a government announcement. There is no single correct output, so we can't assert on an exact string. The slide pattern is: generate with one function, then **ask the LLM whether the output followed the rules** with a second function. The rules here:

1. Under 280 characters (one X/Twitter post).
2. Includes the hashtag `#CityUpdates`.
3. Has a clear, calm, informative tone (no panic, no emojis-only).

In [5]:
POST_RULES = """1. Keep the post under 280 characters.
2. Include the hashtag #CityUpdates at the end.
3. Use a clear, calm, informative tone appropriate for a government agency."""


def write_announcement(topic: str) -> str:
    """Generate a social-media post for a government announcement.

    Indeterminate task: there is no single right answer, so this is tested
    by asking the LLM whether the output followed POST_RULES (see below).
    """
    prompt = f"""You write social-media posts for a city government communications office.
Follow these rules:
{POST_RULES}

Write one post about: {topic}
Output only the post text."""

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.7,
            max_output_tokens=200,
            # Disable thinking so the output budget is spent on the post itself.
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return (response.text or "").strip()


def post_follows_rules(post: str) -> str:
    """LLM-as-judge: does the post follow POST_RULES? Returns 'Yes' or 'No'."""
    prompt = f"""Does the social-media post below follow ALL of these rules?
{POST_RULES}

Answer with ONLY the single word Yes or No.

Post: {post}
Answer:"""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=5,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return (response.text or "").strip().rstrip(".").capitalize()


# Quick smoke test
example = write_announcement("snow emergency: city offices closed Friday")
print(example)
print("Length:", len(example))
print("Follows rules?", post_follows_rules(example))

Due to the snow emergency, all city offices will be closed tomorrow, Friday, February 2nd. Essential services will continue. Stay safe and warm! #CityUpdates
Length: 157
Follows rules? Yes


## 4 · Unit tests with `pytest`

We use [`ipytest`](https://github.com/chmp/ipytest) to run `pytest` inside the notebook. Two styles, matching the slides:

- **Deterministic** (`classify_question`): assert the exact expected category.
- **Indeterminate** (`write_announcement`): generate, then assert the LLM judge says the rules were followed; also include a negative case where a clearly non-compliant post should be judged `No`.

In [6]:
import ipytest

ipytest.autoconfig()

In [7]:
%%ipytest -v

import pytest


# --- Deterministic function: classify_question -----------------------------

@pytest.mark.parametrize(
    "question,expected",
    [
        ("How do I apply for an open position at the city?", "Employment"),
        ("My house is on fire, who do I call?", "Emergency Services"),
        ("When is the deadline to pay my property taxes?", "Tax Related"),
        ("What time does city hall open on Mondays?", "General Information"),
    ],
)
def test_classify_question(question, expected):
    assert classify_question(question) == expected


# --- Indeterminate function: write_announcement ----------------------------

def test_announcement_follows_rules():
    """A generated post should be judged as following the rules."""
    post = write_announcement("reminder: leaf pickup begins next Monday")
    assert post_follows_rules(post) == "Yes"


def test_announcement_under_char_limit():
    """Hard, deterministic check on the generated post's length."""
    post = write_announcement("city pools open for the summer season")
    assert len(post) < 280


def test_judge_rejects_bad_post():
    """Negative case: a post that breaks the rules should be judged 'No'."""
    bad_post = "City offices closed."  # no hashtag -> violates rule 2
    assert post_follows_rules(bad_post) == "No"

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-9.1.0, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 7 items

t_02b3bfb6ef734745986979509c3b2fe0.py .......                                                [100%]

========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=================================== 7 passed, 1 warning in 4.14s ===================================


## 5 · Compare two prompts with the Gen AI Evaluation Service

Unit tests tell us *pass/fail*; the Evaluation API tells us *how good* and lets us **compare prompts** at scale. Here we A/B two prompts for the announcement task:

- **Prompt A (terse):** minimal instructions.
- **Prompt B (detailed):** the full rules from Section 3.

We run a **pointwise** evaluation on each prompt over the same set of announcement topics, scoring with model-based metrics (`fluency`, `coherence`, `instruction_following`, `verbosity`), then compare the summary scores side by side.

The evaluation dataset uses the `prompt` column; we let the `EvalTask` call the model for each row so we are genuinely comparing the *prompts*, not pre-generated text.

In [8]:
import datetime

import pandas as pd
import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples
from vertexai.generative_models import GenerativeModel

# The Evaluation Service lives in the vertexai SDK; initialize it.
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Topics we'll generate an announcement for, used by BOTH prompt variants.
TOPICS = [
    "snow emergency: all city offices closed Friday",
    "public schools closed tomorrow due to a winter storm",
    "Independence Day fireworks at the waterfront, 9pm Saturday",
    "boil-water advisory lifted for the downtown district",
    "reminder: property tax payments due by the end of the month",
]

PROMPT_A = "Write a social media post for a city about: {topic}"

PROMPT_B = (
    "You write social-media posts for a city government communications office.\n"
    "Follow these rules:\n" + POST_RULES + "\n\n"
    "Write one post about: {topic}\nOutput only the post text."
)


def build_dataset(prompt_template: str) -> pd.DataFrame:
    """Build an eval dataset whose 'prompt' column is the rendered prompt."""
    return pd.DataFrame({"prompt": [prompt_template.format(topic=t) for t in TOPICS]})


dataset_a = build_dataset(PROMPT_A)
dataset_b = build_dataset(PROMPT_B)
dataset_a

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


,prompt
0,Write a social media post for a city about: sn...
1,Write a social media post for a city about: pu...
2,Write a social media post for a city about: In...
3,Write a social media post for a city about: bo...
4,Write a social media post for a city about: re...


In [10]:
# Model-based metrics from the Gen AI Evaluation Service (see slides).
METRICS = [
    MetricPromptTemplateExamples.Pointwise.FLUENCY,
    MetricPromptTemplateExamples.Pointwise.COHERENCE,
    MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
    MetricPromptTemplateExamples.Pointwise.VERBOSITY,
]

# The EvalTask generates a response for each prompt using this model.
eval_model = GenerativeModel(MODEL_ID)

run_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")


def run_eval(dataset: pd.DataFrame, label: str):
    task = EvalTask(
        dataset=dataset,
        metrics=METRICS,
        experiment="challenge-03-announcement-prompts",
    )
    return task.evaluate(
        model=eval_model,
        experiment_run_name=f"{label}-{run_ts}",
    )


result_a = run_eval(dataset_a, "prompt-a-terse")

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 5 responses from Gemini model gemini-2.5-flash.
100%|██████████| 5/5 [00:12<00:00,  2.50s/it]
INFO:vertexai.evaluation._evaluation:All 5 responses are successfully generated from Gemini model gemini-2.5-flash.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 12.512039266999636 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:34<00:00,  1.74s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:34.85715420800261 seconds


In [11]:
# To prevent 429 errors, wait ~1 minute before running evals on set b
time.sleep(60)

result_b = run_eval(dataset_b, "prompt-b-detailed")
print("Evaluations complete.")

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 5 responses from Gemini model gemini-2.5-flash.
100%|██████████| 5/5 [00:02<00:00,  1.81it/s]
INFO:vertexai.evaluation._evaluation:All 5 responses are successfully generated from Gemini model gemini-2.5-flash.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 2.774552724997193 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:24<00:00,  1.23s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:24.541807428999164 seconds


Evaluations complete.


In [12]:
# Compare the two prompts side by side on their summary metrics.
comparison = pd.DataFrame(
    {
        "Prompt A (terse)": result_a.summary_metrics,
        "Prompt B (detailed)": result_b.summary_metrics,
    }
)
comparison

,Prompt A (terse),Prompt B (detailed)
row_count,5.000000,5.0
fluency/mean,5.000000,5.0
fluency/std,0.000000,0.0
coherence/mean,5.000000,5.0
coherence/std,0.000000,0.0
instruction_following/mean,5.000000,5.0
instruction_following/std,0.000000,0.0
verbosity/mean,0.800000,0.0
verbosity/std,0.447214,0.0


In [13]:
# Per-row detail for the b prompt, so we can eyeball the actual responses.
result_b.metrics_table

,prompt,response,fluency/explanation,fluency/score,coherence/explanation,coherence/score,instruction_following/explanation,instruction_following/score,verbosity/explanation,verbosity/score
0,You write social-media posts for a city govern...,"Due to the snow emergency, all city offices wi...",The response is completely fluent. It is free ...,5.0,"The response is completely coherent, presentin...",5.0,The response completely fulfills all instructi...,5.0,"The response is perfectly concise, providing a...",0.0
1,You write social-media posts for a city govern...,"All public schools will be closed tomorrow, [D...","The response is free of grammatical errors, de...",5.0,"The post exhibits seamless logical flow, start...",5.0,The response completely fulfills all instructi...,5.0,"The response is perfectly concise, providing a...",0.0
2,You write social-media posts for a city govern...,Celebrate Independence Day with fireworks at t...,"The response is free of grammatical errors, us...",5.0,The writing demonstrates seamless logical flow...,5.0,The response fully adheres to all instructions...,5.0,"The response is perfectly concise, providing a...",0.0
3,You write social-media posts for a city govern...,The boil-water advisory for the downtown distr...,The response is completely fluent. It is free ...,5.0,The response demonstrates exceptional coherenc...,5.0,The response perfectly adheres to all instruct...,5.0,"The response is perfectly concise, providing a...",0.0
4,You write social-media posts for a city govern...,Friendly reminder: Property tax payments are d...,"The response is free of grammatical errors, us...",5.0,"The post exhibits a seamless logical flow, sta...",5.0,The response perfectly adheres to all instruct...,5.0,"The response is perfectly concise, providing a...",0.0
